# CNN on CIFAR-10: MLP vs CNN and eight optimizers

This notebook contains the final code behind the [CNN CIFAR-10 Optimizer Explorer](../dashboard/) dashboard. It:

1. loads and explores CIFAR-10,
2. trains an MLP baseline and a small CNN, and checks the CNN's parameter count by hand,
3. evaluates the CNN (learning curves, confusion matrix, misclassified images),
4. repeats the MLP vs CNN comparison on Fashion-MNIST for reference,
5. trains the same CNN with eight optimizers and compares them,
6. saves the models and a results table.

Every markdown cell has an Indonesian translation you can expand. Run the notebook top to bottom. On a Colab GPU (*Runtime → Change runtime type → T4 GPU*) the full run takes about 10 minutes.

<details>
<summary>Bahasa Indonesia</summary>

# CNN pada CIFAR-10: MLP vs CNN dan delapan optimizer

Notebook ini berisi kode akhir di balik dashboard [CNN CIFAR-10 Optimizer Explorer](../dashboard/). Isinya:

1. memuat dan mengeksplorasi CIFAR-10,
2. melatih baseline MLP dan CNN kecil, lalu mencocokkan jumlah parameter CNN dengan hitungan manual,
3. mengevaluasi CNN (kurva training, confusion matrix, citra yang salah klasifikasi),
4. mengulang perbandingan MLP vs CNN pada Fashion-MNIST sebagai pembanding,
5. melatih CNN yang sama dengan delapan optimizer lalu membandingkannya,
6. menyimpan model dan tabel hasil.

Setiap sel markdown punya terjemahan bahasa Indonesia yang bisa dibuka seperti ini. Jalankan notebook dari atas ke bawah. Dengan GPU Colab (*Runtime → Change runtime type → T4 GPU*) seluruhnya selesai sekitar 10 menit.

</details>

## 1. Setup

Imports, a fixed random seed so results can be repeated, and a few settings. `EPOCHS` controls every training run; `QUICK_RUN = True` trains on 5,000 images for one epoch, which is useful to check that everything works before the full run.

<details>
<summary>Bahasa Indonesia</summary>

## 1. Setup

Import library, *random seed* tetap agar hasil bisa diulang, dan beberapa pengaturan. `EPOCHS` mengatur semua proses training; `QUICK_RUN = True` hanya melatih 5.000 citra selama satu epoch, berguna untuk mengecek semuanya berjalan sebelum run penuh.

</details>

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

SEED = 42
EPOCHS = 10
BATCH_SIZE = 128
QUICK_RUN = False          # True: 5,000 training images and 1 epoch, for a fast check

if QUICK_RUN:
    EPOCHS = 1

keras.utils.set_random_seed(SEED)
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "none, training will run on CPU")

## 2. Load CIFAR-10

CIFAR-10 has 60,000 color images of 32 × 32 pixels in 10 classes: 50,000 for training and 10,000 for testing ([dataset page](https://www.cs.toronto.edu/~kriz/cifar.html)). `keras.datasets` downloads it (about 170 MB) the first time.

Pixel values are scaled from 0–255 to 0–1, and the labels are flattened from shape `(N, 1)` to `(N,)`. The images already have a channel axis (3 for RGB), so no reshaping is needed, unlike grayscale Fashion-MNIST.

<details>
<summary>Bahasa Indonesia</summary>

## 2. Memuat CIFAR-10

CIFAR-10 berisi 60.000 citra berwarna 32 × 32 piksel dalam 10 kelas: 50.000 untuk latih dan 10.000 untuk uji ([halaman dataset](https://www.cs.toronto.edu/~kriz/cifar.html)). `keras.datasets` mengunduhnya (sekitar 170 MB) saat pertama kali dijalankan.

Nilai piksel diskalakan dari 0–255 ke 0–1, dan label diratakan dari bentuk `(N, 1)` menjadi `(N,)`. Citra sudah punya sumbu channel (3 untuk RGB), jadi tidak perlu diubah bentuknya seperti Fashion-MNIST yang grayscale.

</details>

In [ ]:
(xc_train, yc_train), (xc_test, yc_test) = keras.datasets.cifar10.load_data()
yc_train, yc_test = yc_train.ravel(), yc_test.ravel()
xc_train = (xc_train / 255.0).astype("float32")
xc_test = (xc_test / 255.0).astype("float32")

if QUICK_RUN:
    xc_train, yc_train = xc_train[:5000], yc_train[:5000]

CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
           "dog", "frog", "horse", "ship", "truck"]
print("train:", xc_train.shape, "| test:", xc_test.shape)

### Explore the data

A grid of training images and the number of images per class. The classes are balanced (5,000 training images each), so accuracy is a fair summary metric here.

<details>
<summary>Bahasa Indonesia</summary>

### Eksplorasi data

Contoh citra latih dan jumlah citra per kelas. Kelasnya seimbang (masing-masing 5.000 citra latih), jadi akurasi adalah ringkasan yang adil untuk data ini.

</details>

In [ ]:
plt.figure(figsize=(8, 8))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.imshow(xc_train[i])
    plt.title(CLASSES[yc_train[i]], fontsize=8)
    plt.axis("off")
plt.suptitle("25 CIFAR-10 training images")
plt.tight_layout()
plt.show()

print("Images per class:", dict(zip(CLASSES, np.bincount(yc_train))))
print("Pixel range:", xc_train.min(), "to", xc_train.max())

## 3. Models

Two models with the same output layer:

- **MLP**: flattens each image into 3,072 numbers and passes them through two dense layers. It treats every pixel as an independent input and ignores where pixels sit relative to each other.
- **CNN**: two convolution + max-pooling blocks, then a dense layer with dropout. Each 3 × 3 kernel is shared across the whole image (*weight sharing*), so the network learns local patterns such as edges with few parameters.

The builders are functions so the optimizer experiment can create a fresh, identical CNN for every run.

<details>
<summary>Bahasa Indonesia</summary>

## 3. Model

Dua model dengan layer output yang sama:

- **MLP**: meratakan setiap citra menjadi 3.072 angka lalu melewatkannya ke dua layer dense. Setiap piksel diperlakukan sebagai input yang berdiri sendiri, tanpa memperhatikan posisinya terhadap piksel lain.
- **CNN**: dua blok konvolusi + max pooling, lalu layer dense dengan dropout. Setiap kernel 3 × 3 dipakai bersama di seluruh citra (*weight sharing*), sehingga jaringan mempelajari pola lokal seperti tepi dengan sedikit parameter.

Model dibuat lewat fungsi supaya eksperimen optimizer bisa membuat CNN baru yang identik untuk setiap run.

</details>

In [ ]:
def build_mlp(input_shape=(32, 32, 3)):
    return keras.Sequential([
        layers.Input(input_shape),
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dense(128, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ], name="mlp")


def build_cnn(input_shape=(32, 32, 3)):
    return keras.Sequential([
        layers.Input(input_shape),
        layers.Conv2D(32, 3, activation="relu", name="conv1"),
        layers.MaxPooling2D(2, name="pool1"),
        layers.Conv2D(64, 3, activation="relu", name="conv2"),
        layers.MaxPooling2D(2, name="pool2"),
        layers.Flatten(name="flatten"),
        layers.Dense(128, activation="relu", name="dense"),
        layers.Dropout(0.3, name="dropout"),
        layers.Dense(10, activation="softmax", name="output"),
    ], name="cnn")


cnn_c = build_cnn()
cnn_c.summary()

### Check the parameter count by hand

A convolution layer has `(K × K × C_in + 1) × C_out` parameters: one weight per kernel cell per input channel, plus one bias per filter. The output size of a 3 × 3 convolution without padding is `W − 3 + 1`, and 2 × 2 pooling halves it (rounding down):

`32 → 30 → 15 → 13 → 6`, so Flatten gives `6 × 6 × 64 = 2,304` values.

The cell below compares these formulas with what Keras reports.

<details>
<summary>Bahasa Indonesia</summary>

### Cek jumlah parameter secara manual

Layer konvolusi punya `(K × K × C_in + 1) × C_out` parameter: satu bobot untuk setiap sel kernel per channel input, ditambah satu bias per filter. Ukuran output konvolusi 3 × 3 tanpa padding adalah `W − 3 + 1`, dan pooling 2 × 2 membaginya dua (dibulatkan ke bawah):

`32 → 30 → 15 → 13 → 6`, sehingga Flatten menghasilkan `6 × 6 × 64 = 2.304` nilai.

Sel di bawah membandingkan rumus ini dengan angka dari Keras.

</details>

In [ ]:
manual = {
    "conv1":   (3 * 3 * 3 + 1) * 32,     # 896, C_in = 3 because the input is RGB
    "pool1":   0,
    "conv2":   (3 * 3 * 32 + 1) * 64,    # 18,496
    "pool2":   0,
    "flatten": 0,
    "dense":   6 * 6 * 64 * 128 + 128,   # 295,040
    "output":  128 * 10 + 10,            # 1,290
}
for name, expected in manual.items():
    layer = cnn_c.get_layer(name)
    actual = layer.count_params()
    status = "ok" if expected == actual else f"MISMATCH, Keras says {actual:,}"
    print(f"{name:8s} output {str(tuple(layer.output.shape)):18s} manual {expected:>8,}  {status}")
print(f"\nTotal CNN parameters: {cnn_c.count_params():,}")
print(f"Total MLP parameters: {build_mlp().count_params():,}")

## 4. Train the MLP and the CNN

Both models use Adam, sparse categorical cross-entropy (labels are integers 0–9), and keep the last 10% of the training data for validation. `results` collects the test accuracy, parameter count, and training time of every run in this notebook.

<details>
<summary>Bahasa Indonesia</summary>

## 4. Melatih MLP dan CNN

Kedua model memakai Adam, sparse categorical cross-entropy (labelnya bilangan bulat 0–9), dan menyisihkan 10% terakhir data latih untuk validasi. `results` mengumpulkan akurasi uji, jumlah parameter, dan waktu latih dari setiap run di notebook ini.

</details>

In [ ]:
results = {}


def train(model, x, y, x_test, y_test, optimizer="adam", label=None, verbose=2):
    """Compile, fit, evaluate, and record one run."""
    model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    t0 = time.time()
    history = model.fit(x, y, epochs=EPOCHS, batch_size=BATCH_SIZE,
                        validation_split=0.1, verbose=verbose)
    seconds = time.time() - t0
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    results[label or model.name] = {"params": model.count_params(), "test_accuracy": round(test_acc, 4),
                                    "test_loss": round(test_loss, 4), "train_seconds": round(seconds, 1)}
    print(f"{label or model.name}: test accuracy {test_acc:.4f}, {seconds:.0f} s")
    return history


keras.utils.set_random_seed(SEED)
mlp_c = build_mlp()
hist_mlp = train(mlp_c, xc_train, yc_train, xc_test, yc_test, label="CIFAR-10 MLP")

keras.utils.set_random_seed(SEED)
cnn_c = build_cnn()
hist_cnn = train(cnn_c, xc_train, yc_train, xc_test, yc_test, label="CIFAR-10 CNN (Adam)")

### Learning curves

Loss and accuracy per epoch for training (solid) and validation (dashed). A validation curve that flattens or turns upward while the training curve keeps improving is a sign of overfitting.

<details>
<summary>Bahasa Indonesia</summary>

### Kurva training

Loss dan akurasi per epoch untuk data latih (garis penuh) dan validasi (garis putus-putus). Kurva validasi yang mendatar atau naik kembali sementara kurva latih terus membaik adalah tanda overfitting.

</details>

In [ ]:
def plot_histories(histories):
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    for name, h in histories.items():
        epochs = range(1, len(h.history["loss"]) + 1)
        line, = ax[0].plot(epochs, h.history["loss"], marker="o", ms=3, label=f"{name} train")
        ax[0].plot(epochs, h.history["val_loss"], "--", color=line.get_color(), label=f"{name} val")
        ax[1].plot(epochs, h.history["accuracy"], marker="o", ms=3, color=line.get_color(), label=f"{name} train")
        ax[1].plot(epochs, h.history["val_accuracy"], "--", color=line.get_color(), label=f"{name} val")
    ax[0].set_title("Loss"); ax[1].set_title("Accuracy")
    for a in ax:
        a.set_xlabel("Epoch"); a.grid(alpha=0.3); a.legend(fontsize=8)
    plt.show()


plot_histories({"MLP": hist_mlp, "CNN": hist_cnn})

## 5. Evaluate the CNN

The confusion matrix counts, for every true class (row), how often the CNN predicted each class (column). The diagonal holds correct predictions. The cell after it lists the class pairs the model confuses most and the accuracy of each class.

<details>
<summary>Bahasa Indonesia</summary>

## 5. Evaluasi CNN

Confusion matrix menghitung, untuk setiap kelas asli (baris), seberapa sering CNN memprediksi setiap kelas (kolom). Diagonal berisi prediksi yang benar. Sel berikutnya menampilkan pasangan kelas yang paling sering tertukar dan akurasi tiap kelas.

</details>

In [ ]:
prob_c = cnn_c.predict(xc_test, verbose=0)
yc_pred = prob_c.argmax(axis=1)
cm_c = confusion_matrix(yc_test, yc_pred)

fig, ax = plt.subplots(figsize=(9, 9))
ConfusionMatrixDisplay(cm_c, display_labels=CLASSES).plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
ax.set_title("CNN confusion matrix, CIFAR-10 test set")
plt.show()

In [ ]:
off_diag = cm_c.copy()
np.fill_diagonal(off_diag, 0)
print("Most confused pairs (true -> predicted):")
for k in np.argsort(off_diag, axis=None)[::-1][:5]:
    i, j = np.unravel_index(k, cm_c.shape)
    print(f"  {CLASSES[i]:10s} -> {CLASSES[j]:10s} {off_diag[i, j]} images")

print("\nAccuracy per class:")
for i, name in enumerate(CLASSES):
    print(f"  {name:10s} {cm_c[i, i] / cm_c[i].sum():.3f}")

### Misclassified images

The first 15 test images the CNN got wrong, with the true label, the prediction, and the model's confidence (the highest softmax probability). Many of these are hard even for a person at 32 × 32 pixels: animals in unusual poses, cluttered backgrounds, or cars photographed from angles that look like trucks.

<details>
<summary>Bahasa Indonesia</summary>

### Citra yang salah klasifikasi

15 citra uji pertama yang salah ditebak CNN, beserta label asli, prediksi, dan keyakinan model (probabilitas softmax tertinggi). Banyak di antaranya sulit dikenali manusia sekalipun pada 32 × 32 piksel: hewan dengan pose tidak biasa, latar belakang ramai, atau mobil yang difoto dari sudut yang mirip truk.

</details>

In [ ]:
wrong = np.where(yc_pred != yc_test)[0]
print(f"Misclassified: {len(wrong)} of {len(yc_test)}")

plt.figure(figsize=(12, 7))
for i, idx in enumerate(wrong[:15]):
    plt.subplot(3, 5, i + 1)
    plt.imshow(xc_test[idx])
    plt.title(f"true: {CLASSES[yc_test[idx]]}\npred: {CLASSES[yc_pred[idx]]}\n"
              f"confidence {prob_c[idx].max():.0%}", fontsize=8)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 6. Reference: the same comparison on Fashion-MNIST

[Fashion-MNIST](https://github.com/zalandoresearch/fashion-mnist) has 70,000 grayscale 28 × 28 images of clothing. Training the same two architectures on it shows how much the gap between MLP and CNN depends on the data. Grayscale images need an explicit channel axis, so they are reshaped to `(N, 28, 28, 1)`.

<details>
<summary>Bahasa Indonesia</summary>

## 6. Pembanding: perbandingan yang sama pada Fashion-MNIST

[Fashion-MNIST](https://github.com/zalandoresearch/fashion-mnist) berisi 70.000 citra pakaian grayscale berukuran 28 × 28. Melatih dua arsitektur yang sama pada data ini memperlihatkan seberapa besar selisih MLP dan CNN bergantung pada datanya. Citra grayscale perlu sumbu channel eksplisit, jadi bentuknya diubah menjadi `(N, 28, 28, 1)`.

</details>

In [ ]:
(xf_train, yf_train), (xf_test, yf_test) = keras.datasets.fashion_mnist.load_data()
xf_train = (xf_train / 255.0).astype("float32")[..., None]
xf_test = (xf_test / 255.0).astype("float32")[..., None]
if QUICK_RUN:
    xf_train, yf_train = xf_train[:5000], yf_train[:5000]

keras.utils.set_random_seed(SEED)
train(build_mlp((28, 28, 1)), xf_train, yf_train, xf_test, yf_test, label="Fashion-MNIST MLP", verbose=0)
keras.utils.set_random_seed(SEED)
train(build_cnn((28, 28, 1)), xf_train, yf_train, xf_test, yf_test, label="Fashion-MNIST CNN (Adam)", verbose=0)

comparison = pd.DataFrame(results).T.astype({"params": int})
display(comparison)

On Fashion-MNIST the CNN wins by a few points. On CIFAR-10 the gap is much larger, because color images with varied backgrounds need local, position-independent features that an MLP cannot learn efficiently. The CNN also uses fewer parameters than the MLP on both datasets.

<details>
<summary>Bahasa Indonesia</summary>

Di Fashion-MNIST, CNN unggul beberapa poin. Di CIFAR-10 selisihnya jauh lebih besar, karena citra berwarna dengan latar yang beragam membutuhkan fitur lokal yang tidak bergantung posisi, dan itu tidak bisa dipelajari MLP secara efisien. CNN juga memakai parameter lebih sedikit daripada MLP di kedua dataset.

</details>

## 7. Optimizer experiment

The same CNN is trained again with eight optimizers. Architecture, seed (so the starting weights are identical), epochs, batch size, and validation split stay fixed, so any difference comes from the optimizer.

| Optimizer | How it updates the weights | When to use it |
|---|---|---|
| SGD | Plain step against the gradient | Learning the basics, building a baseline |
| RMSprop | Scales the learning rate by a running average of squared gradients | An alternative for training neural networks |
| Adam | Momentum plus adaptive per-parameter scaling | A practical first choice for CNNs |
| AdamW | Adam with decoupled weight decay | When you want weight regularization |
| Adagrad | Step size shrinks with accumulated squared gradients | Sparse features or rare gradients |
| Nadam | Adam with Nesterov-style momentum | An Adam alternative for experiments |
| Adadelta | Adagrad variant that limits the accumulation | Experiments with adaptive methods |
| SGD + momentum | SGD that keeps part of the previous update (lr 0.05, momentum 0.9, Nesterov) | Extra run: can SGD catch up with momentum? |

All optimizers except the last use Keras default learning rates. SGD (0.01, no momentum), Adagrad, and Adadelta (0.001) learn slowly with these defaults; that is part of what the experiment shows.

<details>
<summary>Bahasa Indonesia</summary>

## 7. Eksperimen optimizer

CNN yang sama dilatih ulang dengan delapan optimizer. Arsitektur, seed (sehingga bobot awalnya identik), epoch, batch size, dan pembagian validasi dibuat tetap, jadi perbedaan hasil hanya berasal dari optimizer.

| Optimizer | Karakteristik | Kapan digunakan? |
|---|---|---|
| SGD | Pembaruan sederhana berdasarkan gradien | Memahami dasar optimisasi dan membuat baseline |
| RMSprop | Menyesuaikan learning rate berdasarkan riwayat kuadrat gradien | Alternatif untuk pelatihan jaringan saraf |
| Adam | Menggabungkan momentum dan penskalaan adaptif dari gradien | Pilihan awal yang praktis untuk CNN |
| AdamW | Adam dengan decoupled weight decay | Ketika ingin menambahkan regularisasi bobot |
| Adagrad | Menyesuaikan langkah berdasarkan akumulasi kuadrat gradien | Masalah dengan fitur atau gradien yang jarang muncul |
| Nadam | Adam dengan momentum bergaya Nesterov | Alternatif Adam untuk eksperimen |
| Adadelta | Metode adaptif yang membatasi akumulasi gradien Adagrad | Eksperimen optimisasi adaptif |
| SGD + momentum | SGD yang membawa sebagian pembaruan sebelumnya (lr 0,05, momentum 0,9, Nesterov) | Run tambahan: apakah SGD bisa menyusul dengan momentum? |

Semua optimizer kecuali yang terakhir memakai learning rate bawaan Keras. SGD (0,01, tanpa momentum), Adagrad, dan Adadelta (0,001) belajar lambat dengan nilai bawaan ini; itu bagian dari temuan eksperimen.

</details>

In [ ]:
OPTIMIZERS = {
    "SGD":          lambda: keras.optimizers.SGD(),
    "RMSprop":      lambda: keras.optimizers.RMSprop(),
    "Adam":         lambda: keras.optimizers.Adam(),
    "AdamW":        lambda: keras.optimizers.AdamW(),
    "Adagrad":      lambda: keras.optimizers.Adagrad(),
    "Nadam":        lambda: keras.optimizers.Nadam(),
    "Adadelta":     lambda: keras.optimizers.Adadelta(),
    "SGD momentum": lambda: keras.optimizers.SGD(learning_rate=0.05, momentum=0.9, nesterov=True),
}

opt_histories, opt_models, rows = {}, {}, []
for name, make_optimizer in OPTIMIZERS.items():
    keras.utils.set_random_seed(SEED)
    model = build_cnn()
    h = train(model, xc_train, yc_train, xc_test, yc_test,
              optimizer=make_optimizer(), label=f"CIFAR-10 CNN ({name})", verbose=0)
    opt_histories[name], opt_models[name] = h, model
    val = h.history["val_accuracy"]
    r = results[f"CIFAR-10 CNN ({name})"]
    rows.append({"optimizer": name, "test accuracy": r["test_accuracy"], "test loss": r["test_loss"],
                 "best val accuracy": max(val), "best epoch": int(np.argmax(val)) + 1,
                 "train-val gap": h.history["accuracy"][-1] - val[-1], "seconds": r["train_seconds"]})

df_opt = pd.DataFrame(rows).set_index("optimizer").sort_values("test accuracy", ascending=False)
display(df_opt.round(4))

### Compare the optimizers

Validation curves for all eight runs, then test accuracy as a bar chart. The dashed line in the bar chart is the MLP for reference.

<details>
<summary>Bahasa Indonesia</summary>

### Membandingkan optimizer

Kurva validasi untuk kedelapan run, lalu akurasi uji dalam diagram batang. Garis putus-putus di diagram batang adalah MLP sebagai pembanding.

</details>

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
for name, h in opt_histories.items():
    epochs = range(1, len(h.history["loss"]) + 1)
    ax[0].plot(epochs, h.history["val_loss"], marker="o", ms=3, label=name)
    ax[1].plot(epochs, h.history["val_accuracy"], marker="o", ms=3, label=name)
ax[0].set_title("Validation loss"); ax[1].set_title("Validation accuracy")
for a in ax:
    a.set_xlabel("Epoch"); a.grid(alpha=0.3); a.legend(fontsize=8)
plt.show()

order = df_opt["test accuracy"].sort_values()
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(order.index, order.values, color="#2563EB")
ax.axvline(results["CIFAR-10 MLP"]["test_accuracy"], ls="--", color="grey", label="MLP (Adam)")
for i, v in enumerate(order.values):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=9)
ax.set_xlim(0, order.max() + 0.08)
ax.set_xlabel("Test accuracy")
ax.set_title(f"CNN test accuracy per optimizer ({EPOCHS} epochs)")
ax.legend(loc="lower right")
plt.show()

In [ ]:
# Train vs validation for the three best optimizers, to look for overfitting
plot_histories({name: opt_histories[name] for name in df_opt.index[:3]})

### What the results show

From the run published in the dashboard (numbers in your run will differ slightly):

- **Nadam, RMSprop, Adam, and AdamW** finish within about 1.2 points of each other (around 68% test accuracy). With one seed, that difference is too small to call one of them better.
- **SGD, Adagrad, and Adadelta** stay far behind (roughly 47%, 32%, and 16%) because their default learning rates are too small for 10 epochs. Adadelta's predictions are barely more confident than a uniform guess.
- **SGD with momentum** and a larger learning rate closes most of the gap (about 64%), but it has the largest train−val gap, so it starts to overfit sooner.
- The adaptive optimizers that work well reach their best validation accuracy at the last epoch, so more epochs would likely still help.

<details>
<summary>Bahasa Indonesia</summary>

### Apa yang ditunjukkan hasilnya

Dari run yang dipublikasikan di dashboard (angka di run-mu akan sedikit berbeda):

- **Nadam, RMSprop, Adam, dan AdamW** berakhir dalam rentang sekitar 1,2 poin satu sama lain (akurasi uji sekitar 68%). Dengan satu seed, selisih sekecil itu belum cukup untuk menyebut salah satunya lebih baik.
- **SGD, Adagrad, dan Adadelta** tertinggal jauh (sekitar 47%, 32%, dan 16%) karena learning rate bawaannya terlalu kecil untuk 10 epoch. Prediksi Adadelta hampir tidak lebih yakin daripada tebakan acak.
- **SGD dengan momentum** dan learning rate lebih besar menutup sebagian besar ketertinggalan (sekitar 64%), tetapi selisih train−val-nya paling besar, jadi lebih cepat mulai overfitting.
- Optimizer adaptif yang bekerja baik mencapai akurasi validasi terbaiknya di epoch terakhir, jadi menambah epoch kemungkinan masih membantu.

</details>

## 8. Save models and results

Saves every trained CNN as `models/cnn-<optimizer>.keras` with its history, the MLP as `models/mlp.keras`, and the summary table as `results/optimizer_results.csv`. These are the files `src/export_dashboard.py` reads to rebuild the dashboard data. In Colab, download them from the file panel or copy them to Google Drive.

<details>
<summary>Bahasa Indonesia</summary>

## 8. Menyimpan model dan hasil

Menyimpan setiap CNN hasil latihan sebagai `models/cnn-<optimizer>.keras` beserta riwayatnya, MLP sebagai `models/mlp.keras`, dan tabel ringkasan sebagai `results/optimizer_results.csv`. Berkas-berkas inilah yang dibaca `src/export_dashboard.py` untuk membangun ulang data dashboard. Di Colab, unduh lewat panel file atau salin ke Google Drive.

</details>

In [ ]:
models_dir, results_dir = Path("models"), Path("results")
models_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

def slug(name):
    return name.lower().replace(" ", "-")

for name, model in opt_models.items():
    model.save(models_dir / f"cnn-{slug(name)}.keras")
    json.dump({"name": name, "time": results[f"CIFAR-10 CNN ({name})"]["train_seconds"],
               "hist": {k: [round(float(v), 4) for v in vs] for k, vs in opt_histories[name].history.items()}},
              open(models_dir / f"cnn-{slug(name)}.history.json", "w"), indent=1)

mlp_c.save(models_dir / "mlp.keras")
json.dump({"name": "MLP", "time": results["CIFAR-10 MLP"]["train_seconds"],
           "hist": {k: [round(float(v), 4) for v in vs] for k, vs in hist_mlp.history.items()}},
          open(models_dir / "mlp.history.json", "w"), indent=1)

summary = pd.DataFrame(results).T.astype({"params": int})
summary.to_csv(results_dir / "optimizer_results.csv", index_label="run")
display(summary)
print("Saved to", models_dir.resolve(), "and", results_dir.resolve())